# Figure 1 teaser (plot-only)

Set paths in the next cell (`CUBE_DIR`, etc.). No shell `export` required.

Mosaic inference is **not** run here. Precomputed reconstructions ship in
`evaluation/maryland_evals/`; to regenerate them on GPU:

```powershell
# Windows (PowerShell)
python scripts/run_maryland_mosaic.py `
  --model-root C:\path\to\lidar_chm_models `
  --cube-dir C:\path\to\Maryland_test_cube_folder `
  --save-root evaluation\maryland_evals
```

```bash
# Linux (GEMSR_MODEL_ROOT / GEMSR_CUBE_DIR env vars also work)
python scripts/run_maryland_mosaic.py \
  --model-root /path/to/lidar_chm_models \
  --cube-dir samples/test_cubes
```

Overlay FPS/FID/RMSE/top-1% come from locked Maryland **test-split** tables (`figures.overlays`).

In [ ]:
import importlib
import os
import sys
from pathlib import Path

def _repo_root() -> Path:
    for candidate in (Path.cwd(), Path.cwd() / "GeM-SR"):
        if (candidate / "benchmark_hub.py").is_file():
            return candidate.resolve()
    return Path.cwd().resolve()

REPO_ROOT = _repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import figures.overlays as overlays
import figures.plot_teaser as plot_teaser_mod
importlib.reload(overlays)
importlib.reload(plot_teaser_mod)

from figures.chm_pair import extract_chm_pair
from figures.cube import load_hhdc_cube, resolve_cube_dir
from figures.overlays import MODEL_METRICS, SR3_LINE, DDPM_SCATTER
from figures.plot_teaser import missing_mosaic_files, plot_teaser_from_disk

assert "rmse" in DDPM_SCATTER, "overlays.py missing DDPM rmse; re-run this cell"

# --- Edit these paths (Windows-friendly; no export needed) ---
CUBE_DIR = REPO_ROOT / "samples" / "test_cubes"  # ships with the repo; or your own folder
CUBE_NPZ = None  # or direct path: Path(r"C:\data\Maryland_test_cube.npz")
EVAL_ROOT = REPO_ROOT / "evaluation" / "maryland_evals"
TEASER_LOCAL = REPO_ROOT / "evaluation" / "teaser.pdf"
TEASER_TGRS = REPO_ROOT.parent / "Paper_for_TGRS" / "figures" / "teaser.pdf"
# Optional env fallback if CUBE_DIR is None:
# os.environ["GEMSR_CUBE_DIR"] = r"C:\data\Maryland_test_cube_folder"

In [ ]:
# Load LR/HR CHM from Maryland HHDC cube (no GPU)
if CUBE_NPZ is not None:
    cube_path = Path(CUBE_NPZ)
elif CUBE_DIR is not None:
    cube_path = Path(CUBE_DIR) / "Maryland_test_cube.npz"
else:
    cube_path = resolve_cube_dir() / "Maryland_test_cube.npz"

if not cube_path.is_file():
    raise FileNotFoundError(
        f"{cube_path} not found. Set CUBE_DIR / CUBE_NPZ above."
    )

cube = load_hhdc_cube(cube_path, array_key="arr_0")
pair = extract_chm_pair(cube)
lr_chm = pair["lr_chm"]
hr_chm = pair["hr_chm"]
print(f"LR CHM {lr_chm.shape}, HR CHM {hr_chm.shape}")

In [ ]:
# Plot Figure 1 from saved mosaic .npy + locked overlays
missing = missing_mosaic_files(EVAL_ROOT)
if missing:
    print("STOP: mosaic recon .npy not in this repo.")
    print("Restore evaluation/maryland_evals/ from the repo, or regenerate on GPU:")
    print("  python scripts/run_maryland_mosaic.py --model-root ... --cube-dir ...")
    for path in missing:
        print(f"  missing: {path}")
else:
    copy_dest = TEASER_TGRS if TEASER_TGRS.parent.is_dir() else None
    plot_teaser_from_disk(
        lr=lr_chm / 128.0,
        hr=hr_chm / 128.0,
        eval_root=EVAL_ROOT,
        save_path=TEASER_LOCAL,
        copy_to=copy_dest,
        hr_zoom=(48, 96, 48, 48),
        y_metric="rmse",
        show=True,
    )
    print(f"Saved: {TEASER_LOCAL}")
    if copy_dest is not None:
        print(f"Copied: {copy_dest}")